# OmniVoice Studio — Kaggle Dual-T4 + AI-native Server

Optimized for Kaggle sessions exposing **2× Tesla T4 (~15 GiB each), ~30 GiB RAM, and local SSD**.

```text
cuda:0  → OmniVoice TTS
cuda:1  → Whisper ASR verification
CPU/RAM → preprocessing, API/MCP, Gradio, file I/O
SSD     → /kaggle/working/OmniVoiceStudio + transient caches
```

For dual-GPU sessions this mapping is explicit. The hardware detector advises the quality preset, but it cannot leave the second T4 idle because of a stale notebook import.


In [ ]:
# Kaggle Internet is needed only on a cold path or when the exact code revision changed.
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

CACHE_VERSION = "v1"
MODEL_ID = "k2-fsa/OmniVoice"
ASR_MODEL = "openai/whisper-small.en"
WORKSPACE = "/kaggle/working/OmniVoiceStudio"
LOCAL_CACHE_BASE = Path("/kaggle/working/.cache/omnivoice")
ATTACHED_CACHE_BASE = Path("/kaggle/input/omnivoice-startup-cache")
CACHE_EXPORT_BASE = Path("/kaggle/working/OmniVoiceStartupCache")
CACHE_SOURCE_BASE = (
    ATTACHED_CACHE_BASE
    if ATTACHED_CACHE_BASE.exists()
    else (CACHE_EXPORT_BASE if CACHE_EXPORT_BASE.exists() else None)
)

Path(WORKSPACE).mkdir(parents=True, exist_ok=True)
LOCAL_CACHE_BASE.mkdir(parents=True, exist_ok=True)

compatibility = {
    "schema_version": 1,
    "cache_version": CACHE_VERSION,
    "python_version": f"{sys.version_info.major}.{sys.version_info.minor}",
    "system": platform.system().lower(),
    "machine": platform.machine().lower(),
}
cache_payload = json.dumps(compatibility, sort_keys=True, separators=(",", ":"))
CACHE_KEY = f"{CACHE_VERSION}-" + hashlib.sha256(cache_payload.encode()).hexdigest()[:16]
LOCAL_CACHE = LOCAL_CACHE_BASE / CACHE_KEY
SOURCE_CACHE = CACHE_SOURCE_BASE / CACHE_KEY if CACHE_SOURCE_BASE else None
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

def _read_metadata(path):
    if path is None:
        return {}
    try:
        value = json.loads((path / "metadata.json").read_text(encoding="utf-8"))
        return value if isinstance(value, dict) else {}
    except Exception:
        return {}

source_metadata = _read_metadata(SOURCE_CACHE)
RESOURCE_FAST_PATH = source_metadata.get("compatibility") == compatibility
cached_ref = source_metadata.get("last_package_ref")

try:
    with urllib.request.urlopen(
        "https://api.github.com/repos/binhminhanh1235/OmniVoice/branches/master",
        timeout=15,
    ) as response:
        PACKAGE_REF = json.load(response)["commit"]["sha"]
except Exception as exc:
    if cached_ref:
        PACKAGE_REF = str(cached_ref)
        print("GitHub revision lookup unavailable; using cached revision:", PACKAGE_REF)
    else:
        PACKAGE_REF = "master"
        print("GitHub revision lookup unavailable; cold install will resolve master:", type(exc).__name__)

def _copy_tree(source, destination):
    source = Path(source)
    destination = Path(destination)
    if source.exists():
        shutil.copytree(
            source,
            destination,
            dirs_exist_ok=True,
            symlinks=False,
            ignore=shutil.ignore_patterns("*.tmp", ".nfs*"),
        )

if RESOURCE_FAST_PATH and SOURCE_CACHE is not None:
    _copy_tree(SOURCE_CACHE / "pip", LOCAL_CACHE / "pip")
    _copy_tree(SOURCE_CACHE / "wheels", LOCAL_CACHE / "wheels")

os.environ["PIP_CACHE_DIR"] = str(LOCAL_CACHE / "pip")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

package_key = hashlib.sha256(PACKAGE_REF.encode()).hexdigest()[:16]
wheel_dir = LOCAL_CACHE / "wheels" / package_key
wheel_dir.mkdir(parents=True, exist_ok=True)
wheels = sorted(wheel_dir.glob("omnivoice-*.whl"))

if wheels:
    WHEEL_FAST_PATH = True
    wheel = wheels[-1]
else:
    WHEEL_FAST_PATH = False
    subprocess.check_call([
        sys.executable, "-m", "pip", "wheel", "-q", "--no-deps",
        f"git+https://github.com/binhminhanh1235/OmniVoice.git@{PACKAGE_REF}",
        "--wheel-dir", str(wheel_dir),
    ])
    wheels = sorted(wheel_dir.glob("omnivoice-*.whl"))
    if not wheels:
        raise RuntimeError("OmniVoice wheel build completed without producing a wheel.")
    wheel = wheels[-1]

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", str(wheel)
])

os.environ["OMNIVOICE_LOCAL_CACHE_ROOT"] = str(LOCAL_CACHE_BASE)
if CACHE_SOURCE_BASE is not None:
    os.environ["OMNIVOICE_CACHE_SOURCE"] = str(CACHE_SOURCE_BASE)
else:
    os.environ.pop("OMNIVOICE_CACHE_SOURCE", None)
os.environ["OMNIVOICE_CACHE_PERSIST_ROOT"] = str(CACHE_EXPORT_BASE)

from omnivoice.runtime_cache import (
    RuntimeCacheFingerprint,
    apply_cache_environment,
    detect_runtime_cache,
    persist_runtime_cache,
    prepare_runtime_cache,
    write_workspace_cache_metadata,
)

CACHE_FINGERPRINT = RuntimeCacheFingerprint.current(
    cache_version=CACHE_VERSION,
    package_ref=PACKAGE_REF,
)
CACHE_PREPARATION = prepare_runtime_cache(
    detect_runtime_cache(),
    CACHE_FINGERPRINT,
)
apply_cache_environment(CACHE_PREPARATION)

from huggingface_hub import snapshot_download
snapshot_download(MODEL_ID)
snapshot_download(ASR_MODEL)
persist_runtime_cache(CACHE_PREPARATION)
write_workspace_cache_metadata(WORKSPACE, CACHE_PREPARATION)

print("Package revision:", PACKAGE_REF)
print("Resource cache:", "FAST" if CACHE_PREPARATION.fast_path else "COLD")
print("Exact wheel:", "FAST" if WHEEL_FAST_PATH else "BUILT")
print("Local cache:", CACHE_PREPARATION.local_namespace)
print("Cache source:", CACHE_SOURCE_BASE or "none")
print("Cache export:", CACHE_EXPORT_BASE)


In [ ]:
import importlib
import shutil
import torch
import omnivoice.hardware_quality as hardware_quality
from omnivoice.runtime_workspace import detect_runtime_workspace, ensure_runtime_workspace

# Reload after pip upgrade so the current kernel uses the detector from the package now on disk.
hardware_quality = importlib.reload(hardware_quality)

runtime = ensure_runtime_workspace(detect_runtime_workspace())
if runtime.environment != "kaggle":
    raise RuntimeError(f"Expected Kaggle runtime, detected: {runtime.environment}")
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU T4 x2 in Kaggle Notebook settings.")

GPU_COUNT = torch.cuda.device_count()
for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {props.total_memory / 1024**3:.1f} GiB")

hardware = hardware_quality.detect_hardware(device_index=0)
TTS_DEVICE = "cuda:0"
ASR_DEVICE = "cuda:1" if GPU_COUNT >= 2 else hardware.recommended_asr_device
ASR_MODEL = "openai/whisper-small.en"

if GPU_COUNT >= 2 and hardware.recommended_asr_device != "cuda:1":
    print("WARNING: detector recommendation differs from dual-GPU runtime; forcing ASR to cuda:1.")

quality_store = hardware_quality.HardwareQualitySettingsStore(WORKSPACE)
if not quality_store.path.exists():
    quality_store.set_default(hardware.recommended_preset)
current_preset = quality_store.load().default_preset

print("Runtime:", runtime.summary())
print("Hardware:", hardware.summary())
for note in hardware.notes:
    print("-", note)
print("OmniVoice device:", TTS_DEVICE)
print("Whisper ASR device:", ASR_DEVICE)
print("Whisper ASR model:", ASR_MODEL)
print("Workspace quality preset:", current_preset)
print("Workspace:", WORKSPACE)
usage = shutil.disk_usage("/kaggle/working")
print(f"Local SSD free: {usage.free / 1024**3:.1f} GiB")


## Why this mapping?

OmniVoice stays entirely on `cuda:0` instead of being sharded across both T4s. `cuda:1` becomes a dedicated Whisper accelerator for chunk verification and optional word timestamps. On a single-GPU session, ASR falls back to the detector recommendation.


## Optional: stable hostname + private access

For ChatGPT / Claude Code / Antigravity, create one remotely-managed Cloudflare Tunnel and map a stable hostname such as `omnivoice.example.com` to `http://localhost:8000`.

Create these Kaggle Secrets when using the stable tunnel:
- `CLOUDFLARE_TUNNEL_TOKEN`
- `OMNIVOICE_API_TOKEN`
- `OMNIVOICE_UI_USERNAME`
- `OMNIVOICE_UI_PASSWORD`


In [ ]:
PUBLIC_URL = "https://omnivoice.example.com"
USE_STABLE_TUNNEL = True

if USE_STABLE_TUNNEL:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    required = {
        "CLOUDFLARE_TUNNEL_TOKEN": secrets.get_secret("CLOUDFLARE_TUNNEL_TOKEN"),
        "OMNIVOICE_API_TOKEN": secrets.get_secret("OMNIVOICE_API_TOKEN"),
        "OMNIVOICE_UI_USERNAME": secrets.get_secret("OMNIVOICE_UI_USERNAME"),
        "OMNIVOICE_UI_PASSWORD": secrets.get_secret("OMNIVOICE_UI_PASSWORD"),
    }
    missing = [name for name, value in required.items() if not value]
    if missing:
        raise RuntimeError("Missing Kaggle Secrets: " + ", ".join(missing))
    for name, value in required.items():
        os.environ[name] = value
    os.environ["OMNIVOICE_API_TOKEN_SCOPES"] = (
        "omnivoice:read,omnivoice:generate,omnivoice:queue,omnivoice:mcp"
    )
    os.environ["OMNIVOICE_PUBLIC_URL"] = PUBLIC_URL
    del required, secrets
    print("Stable private public URL configured:", PUBLIC_URL)


In [ ]:
if USE_STABLE_TUNNEL:
    !wget -q -O /kaggle/working/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod 700 /kaggle/working/cloudflared
    !/kaggle/working/cloudflared --version


## Launch OmniVoice Studio

With the stable tunnel enabled, one hostname exposes `/ui`, `/api/v1`, `/mcp`, and `/health`. If disabled, the notebook falls back to the temporary Gradio share URL.


In [ ]:
try:
    if USE_STABLE_TUNNEL:
        !omnivoice-studio serve \
          --model k2-fsa/OmniVoice \
          --device {TTS_DEVICE} \
          --workspace {WORKSPACE} \
          --asr-model {ASR_MODEL} \
          --asr-device {ASR_DEVICE} \
          --host 0.0.0.0 \
          --port 8000 \
          --tunnel \
          --cloudflared /kaggle/working/cloudflared \
          --public-url {PUBLIC_URL}
    else:
        print("Stable tunnel disabled. Using temporary Gradio share URL.")
        !omnivoice-project-studio \
          --model k2-fsa/OmniVoice \
          --device {TTS_DEVICE} \
          --workspace {WORKSPACE} \
          --asr-model {ASR_MODEL} \
          --asr-device {ASR_DEVICE} \
          --share
finally:
    persist_runtime_cache(CACHE_PREPARATION)
    write_workspace_cache_metadata(WORKSPACE, CACHE_PREPARATION)
    print("Startup/model cache export refreshed:", CACHE_EXPORT_BASE)


## Persistence and cache reuse

Active projects, voices, jobs, checkpoints and WAV files remain under
`/kaggle/working/OmniVoiceStudio`, so they are still ephemeral unless you export
or sync them separately.

Startup resources are different: this notebook writes a reusable cache to
`/kaggle/working/OmniVoiceStartupCache`. Save/version that directory as a
Kaggle Dataset named `omnivoice-startup-cache` and attach it to later sessions.
When attached at `/kaggle/input/omnivoice-startup-cache`, unchanged dependency,
model and Whisper resources take the fast path and are restored to local SSD
before generation starts.
